In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
from langchain.tools import tool

In [12]:
from langchain_openai.chat_models import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.messages.tool import ToolMessage
from langgraph.graph import StateGraph

In [13]:
@tool
def cancel_order(order_id: str) -> str:
    """Cancel an order that hasn't shipped."""
    # (Here you'd call your real backend API)
    return f"Order {order_id} has been cancelled."

In [14]:
def call_model(state):
    msgs = state["messages"]
    order = state.get("order", {"order_id": "UNKNOWN"})

    prompt = (
        f'''You are an ecommerce support agent.
        ORDER ID: {order['order_id']}
        If the customer asks to cancel, call cancel_order(order_id)
        and then send a simple confirmation.
        Otherwise, just respond normally.'''
    )
    full = [SystemMessage(prompt)] + msgs
    
    AIMessage = ChatOpenAI(model="gpt-5", temperature=0)(full)
    out = [first]

    if getattr(first, "tool_calls", None):
        # run the cancel_order tool
        tc = first.tool_calls[0]
        result = cancel_order(**tc["args"])
        out.append(ToolMessage(content=result, tool_call_id=tc["id"]))
        
        # 2nd LLM pass: generate the final confirmation text
        AIMessage = ChatOpenAI(model="gpt-5", temperature=0)(full + out)
        out.append(second)
    return {"messages": out}

In [28]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages

# Define state as a TypedDict
class GraphState(TypedDict):
    order: str | None
    messages: Annotated[list[BaseMessage], add_messages]

def construct_graph():
    g = StateGraph(GraphState)  # Pass the class, not a dict
    g.add_node("assistant", call_model)
    g.set_entry_point("assistant")
    return g.compile()

graph = construct_graph()

In [22]:
if __name__ == "__main__":
    example_order = {"order_id": "A12345"}
    convo = [HumanMessage(content="Please cancel my order A12345.")]
    result = graph.invoke({"order": example_order, "messages": convo})
    
    for msg in result["messages"]:
        print(f"{msg.type}: {msg.content}")

human: Please cancel my order A12345.


In [29]:
example_order = {"order_id": "B73973"}
convo = [HumanMessage(content='''Please cancel order #B73973.
    I found a cheaper option elsewhere.''')]

result = graph.invoke({"order": example_order, "messages": convo})
assert any(("cancel_order" in str(m.content) for m in result["messages"])), "Cancel order tool not called"
assert any(("cancelled" in m.content.lower() for m in result["messages"])), "Confirmation message missing"

print("✅ Agent passed minimal evaluation.")

AssertionError: Cancel order tool not called